Đoạn code này là một phần quan trọng trong quy trình tiền xử lý của mô hình DualGNN. Mục tiêu chính của nó là xây dựng một đồ thị quan hệ Người dùng - Người dùng (User-User Graph) dựa trên việc họ có cùng tương tác với các sản phẩm giống nhau.


In [1]:
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import torch
import pandas as pd
import os
import yaml

Hàm `gen_user_matrix` đóng vai trò cốt lõi trong việc chuẩn bị dữ liệu cho mô hình **DualGNN** bằng cách tạo ra ma trận quan hệ giữa những người dùng (User-User graph) dựa trên các sản phẩm họ dùng chung.

Dưới đây là giải thích chi tiết từng bước trong code:

### 1. Gom nhóm sản phẩm theo người dùng

* **`edge_dict = defaultdict(set)`**: Khởi tạo một từ điển mà mỗi giá trị là một tập hợp (set).
* **Vòng lặp `for edge in all_edge**`: Duyệt qua danh sách tất cả các tương tác (cặp user-item). Với mỗi người dùng, code sẽ thêm mã sản phẩm (`item`) vào tập hợp tương ứng của họ trong `edge_dict`.

### 2. Khởi tạo ma trận rỗng

* **`user_graph_matrix = torch.zeros(num_user, num_user)`**: Tạo một ma trận vuông kích thước $N \times N$ (với $N$ là số lượng người dùng) chứa toàn số 0. Đây sẽ là ma trận kề lưu trữ trọng số quan hệ giữa các người dùng.

### 3. Tính toán độ tương đồng (Sản phẩm chung)

Đoạn này sử dụng hai vòng lặp lồng nhau để so sánh từng cặp người dùng:

* **`key_list.sort()`**: Sắp xếp danh sách ID người dùng để đảm bảo thứ tự duyệt ổn định.
* **Vòng lặp `head` và `rear**`: Duyệt qua các cặp người dùng (A, B) mà không lặp lại (vì vậy vòng lặp thứ hai bắt đầu từ `head + 1`).
* **`inter_len = len(item_head.intersection(item_rear))`**: Đây là dòng quan trọng nhất. Nó sử dụng phép toán tập hợp `intersection` (giao) để tìm xem người dùng A và người dùng B có bao nhiêu món đồ dùng chung.

### 4. Gán trọng số vào ma trận

* **`if inter_len > 0`**: Nếu hai người có ít nhất 1 sản phẩm chung, mối quan hệ này sẽ được ghi nhận.
* **`user_graph_matrix[...][...] = inter_len`**: Gán số lượng sản phẩm chung vào ma trận. Vì mối quan hệ này là hai chiều (A giống B thì B cũng giống A), giá trị được gán đối xứng qua đường chéo chính.

### 5. Theo dõi tiến độ

* **`tqdm`**: Được sử dụng để hiển thị thanh trạng thái (progress bar) trong quá trình tính toán, giúp bạn biết được code đã xử lý được bao nhiêu phần trăm, vì việc so sánh $N^2$ cặp người dùng có thể tốn rất nhiều thời gian.

---

**💡 Lưu ý kỹ thuật:**
Ma trận trả về là một ma trận dày (**Dense Matrix**). Nếu bộ dữ liệu của bạn có hàng chục nghìn người dùng, ma trận này sẽ chiếm rất nhiều RAM. Trong mô hình DualGNN, kết quả này sau đó sẽ được lọc lấy **Top-K** (thường là 200) người dùng tương đồng nhất để tối ưu hóa bộ nhớ trước khi huấn luyện.

Bạn có muốn tôi hướng dẫn cách tích hợp hàm này vào quy trình chạy trên Notebook mà chúng ta đã thảo luận không?

In [ ]:
def gen_user_matrix(all_edge, no_users):
    edge_dict = defaultdict(set)

    for edge in all_edge:
        user, item = edge
        edge_dict[user].add(item)

    min_user = 0
    num_user = no_users
    user_graph_matrix = torch.zeros(num_user, num_user)
    key_list = list(edge_dict.keys())
    key_list.sort()
    bar = tqdm(total=len(key_list))
    for head in range(len(key_list)):
        bar.update(1)
        for rear in range(head + 1, len(key_list)):
            head_key = key_list[head]
            rear_key = key_list[rear]
            item_head = edge_dict[head_key]
            item_rear = edge_dict[rear_key]
            inter_len = len(item_head.intersection(item_rear))
            if inter_len > 0:
                user_graph_matrix[head_key - min_user][rear_key - min_user] = inter_len
                user_graph_matrix[rear_key - min_user][head_key - min_user] = inter_len
    bar.close()
    return user_graph_matrix

1. Hàm chính gen_user_matrix(all_edge, no_users)
   Hàm này tính toán mức độ "thân quen" giữa các người dùng:

Gom nhóm dữ liệu: Đầu tiên, nó duyệt qua danh sách các tương tác (all_edge) và dùng defaultdict(set) để lưu lại danh sách các item mà mỗi người dùng đã mua/xem.

So sánh cặp (Pairwise comparison): Sử dụng hai vòng lặp lồng nhau để so sánh từng cặp người dùng.

Tính toán độ giao thoa: inter_len = len(item_head.intersection(item_rear)) xác định xem hai người dùng có bao nhiêu sản phẩm chung.

Xây dựng ma trận: Nếu có sản phẩm chung, giá trị này được điền vào ma trận đối xứng user_graph_matrix. Đây chính là ma trận kề đại diện cho sức mạnh kết nối giữa các người dùng.


In [ ]:
# --- CẤU HÌNH THAY CHO ARGPARSE ---
dataset_name = "sports"  # Bạn có thể đổi thành 'games', 'baby' tùy ý
print(f"Generating u-u matrix for {dataset_name} ...\n")

# --- XỬ LÝ CONFIG ---
config = {}
# Lưu ý: Kiểm tra kỹ đường dẫn folder của bạn trên Colab
# os.chdir('/content/drive/MyDrive/...')

# Giả sử bạn đang ở thư mục gốc của project, code sẽ tìm vào src/configs
con_dir = "./src/configs"
overall_config_file = os.path.join(con_dir, "overall.yaml")
dataset_config_file = os.path.join(con_dir, "dataset", f"{dataset_name}.yaml")

for file in [overall_config_file, dataset_config_file]:
    if os.path.isfile(file):
        with open(file, "r", encoding="utf-8") as f:
            config.update(yaml.safe_load(f))

# --- ĐỌC DỮ LIỆU ---
dataset_path = os.path.abspath(config["data_path"] + dataset_name)
uid_field = config["USER_ID_FIELD"]
iid_field = config["ITEM_ID_FIELD"]

train_df = pd.read_csv(os.path.join(dataset_path, config["inter_file_name"]), sep="\t")
num_user = len(pd.unique(train_df[uid_field]))

# Chỉ lấy mẫu dương x_label == 0
train_df = train_df[train_df["x_label"] == 0].copy()
train_data = train_df[[uid_field, iid_field]].to_numpy()

# --- TẠO MA TRẬN USER-USER ---
user_graph_matrix = gen_user_matrix(train_data, num_user)
user_graph = user_graph_matrix
user_num = torch.zeros(num_user)
user_graph_dict = {}

# Thống kê số bậc của mỗi nút
for i in range(num_user):
    user_num[i] = len(torch.nonzero(user_graph[i]))

# Lọc Top-200
for i in range(num_user):
    k = int(min(user_num[i], 200))
    if k > 0:
        user_i = torch.topk(user_graph[i], k)
        user_graph_dict[i] = [
            user_i.indices.numpy().tolist(),
            user_i.values.numpy().tolist(),
        ]
    else:
        user_graph_dict[i] = [[], []]

# --- LƯU KẾT QUẢ ---
output_path = os.path.join(dataset_path, config["user_graph_dict_file"])
np.save(output_path, user_graph_dict, allow_pickle=True)
print(f"Done! Saved to {output_path}")